# Clothes segmentation: two fine-tuning experiments on the final model

Both start from the current best model, `final_segformer_b3_e30/best.pt`, which is only read and never changed. Both are judged on the validation split by rules fixed in PLAN.md, section 26.

- **Cell 6, rank 2:** a short fine-tune with weight averaging. About 2 hours.
- **Cell 7, rank 3:** a fine-tune at a larger input, 768×512. About 1.5 hours.

Use the **L4 GPU**. Run the cells from top to bottom; cell 3 asks for `clothseg_code.zip` once. If the session ends, run cells 1 to 5 and the interrupted cell again: it continues from the last finished epoch.

## 1. Settings

In [1]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "11"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600

## 2. Mount Drive and show the environment

In [3]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

Mounted at /content/drive
python       3.13.15 | torch 2.11.0+cu130 | transformers 5.18.0
GPU          NVIDIA L4
GPU memory   22.0 GB
CPU cores    12
local disk   193 GB free
Drive        184 GB free


## 3. Get the code

In [4]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

code version 11 copied from Drive
code ready in /content/clothes-segmentation | version 11


## 4. Install requirements and prepare the dataset

In [5]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.7 MB/s eta 0:00:00
segmentation_models_pytorch 0.5.0 | timm 1.0.30 | opencv 4.14.0
[prepare] cache name: atr_576x384 | local: /content/data/atr_576x384 | Drive archive: /content/drive/MyDrive/clothseg/data/cache/atr_576x384.tar
[prepare] free local disk: 193.3 GB
[prepare] found archive on Drive (1.6 GB). Copying and extracting...
[storage] copied atr_576x384.tar (1.6 GB) -> /content/data in 27s
[storage] extracted atr_576x384.tar -> /content/data in 14s
[prepare] done in 42s
[splits] moved 56 val/test near-duplicates of training images into train. New sizes: train=16038, val=690, test=960


## 5. Helper for comparing a new model with the current best one

Run this once before cells 6 and 7. It evaluates a checkpoint on the validation split at the original image size with flip averaging, and prints it next to the current best model.

In [6]:
import json, subprocess, sys
import pandas as pd

BASE_RUN = "final_segformer_b3_e30"
BASE_CKPT = f"{DRIVE_ROOT}/runs/{BASE_RUN}/checkpoints/best.pt"
CORE = ["upper-clothes", "pants", "skirt", "dress"]
WATCH = CORE + ["belt", "scarf", "sunglasses", "shoes"]
assert os.path.exists(BASE_CKPT), f"The current best model is missing: {BASE_CKPT}"

def val_eval(checkpoint, tag):
    out = f"{DRIVE_ROOT}/reports/eval_{tag}_val_flip"
    print(f"evaluating {tag} on validation ...", flush=True)
    r = subprocess.run([sys.executable, "scripts/evaluate.py", "--checkpoint", checkpoint, "--split", "val", "--out-dir", out, "--flip-tta"],
                       capture_output=True, text=True, cwd=CODE_DIR)
    assert r.returncode == 0, "The evaluation failed. Send me this message: " + (r.stdout + r.stderr)[-3000:]
    return json.load(open(f"{out}/metrics.json")), pd.read_csv(f"{out}/per_class.csv").set_index("class")["iou"]

def compare(checkpoint, tag):
    """Print the new model next to the current best one and say whether it meets the rule fixed in PLAN.md, section 26."""
    assert os.path.exists(checkpoint), f"No checkpoint at {checkpoint}. Did the training cell finish?"
    base_m, base_pc = val_eval(BASE_CKPT, BASE_RUN)
    new_m, new_pc = val_eval(checkpoint, tag)
    rows = {k: (base_m[k], new_m[k]) for k in ("miou", "garment_miou", "core_garment_miou", "clothing_binary_iou", "boundary_miou")}
    rows.update({c: (base_pc[c], new_pc[c]) for c in WATCH})
    df = pd.DataFrame(rows, index=["current best", tag]).T
    df["change"] = df[tag] - df["current best"]
    print()
    print(df.round(4).to_string())
    gain = new_m["garment_miou"] - base_m["garment_miou"]
    worst_core = min(new_pc[c] - base_pc[c] for c in CORE)
    ok = gain >= 0.005 and worst_core >= -0.01
    print()
    print(f"garment mIoU change {100 * gain:+.2f} points; largest drop on a core garment {100 * worst_core:+.2f} points -> "
          + ("MEETS the rule" if ok else "does NOT meet the rule") + " (at least +0.50, and no core garment below -1.00)")

print("helper ready")

helper ready


## 6. Rank 2: short fine-tune with weight averaging

Continues from the current best model in 3 cycles of 3 epochs. Each cycle starts at a quarter of the original learning rate and decays; the weights at the end of each cycle are averaged together with the starting weights. About 2 hours on the L4. The current best model is only read, never changed. If the session ends, run cells 1 to 5 and this cell again.

In [6]:
SWA_RUN = "swa_segformer_b3"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 5 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/finetune.py --drive-root "{DRIVE_ROOT}" --init-checkpoint "{BASE_CKPT}" --run-name {SWA_RUN} --cycles 3 --epochs-per-cycle 3 --time-budget-min {budget}

compare(f"{DRIVE_ROOT}/runs/{SWA_RUN}/checkpoints/best.pt", SWA_RUN)

config.json: 100% 70.0k/70.0k [00:00<00:00, 79.7MB/s]
[model] segformer_b3: 47.2 M parameters (encoder 44.1 M, head 3.2 M) | 15 classes | weights not initialised from ImageNet
[finetune] FRESH START from /content/drive/MyDrive/clothseg/runs/final_segformer_b3_e30/checkpoints/best.pt
[finetune] segformer_b3 | 576x384 | batch 16 | fp16 True | NVIDIA L4 | 8 workers | 3 cycles x 3 epochs | lr x0.25 -> x0.01 per cycle | weight averaging ON
[finetune] data: train 16038, val 690 images | 1002 steps/epoch
[model] segformer_b3: 47.2 M parameters (encoder 44.1 M, head 3.2 M) | 15 classes | weights not initialised from ImageNet
[finetune] cycle 1/3 epoch 1/3 step 50/1002 | loss 0.1317 | lr x0.250 | 19.9 img/s
[finetune] cycle 1/3 epoch 1/3 step 100/1002 | loss 0.1327 | lr x0.249 | 21.5 img/s
[finetune] cycle 1/3 epoch 1/3 step 150/1002 | loss 0.1347 | lr x0.249 | 22.2 img/s
[finetune] cycle 1/3 epoch 1/3 step 200/1002 | loss 0.1364 | lr x0.247 | 22.5 img/s
[finetune] cycle 1/3 epoch 1/3 step 250/

## 7. Rank 3: fine-tune at a larger input (768×512)

It first builds the dataset at 768×512 (several minutes, once), then fine-tunes the current best model for 3 epochs at batch size 8. About 1.5 hours on the L4.

In [7]:
BIG_RUN = "big_segformer_b3_768"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 5 and this cell again.")

os.chdir(CODE_DIR)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height 768 --width 512 --workers 8
budget = int(minutes_left())
!python scripts/finetune.py --drive-root "{DRIVE_ROOT}" --init-checkpoint "{BASE_CKPT}" --run-name {BIG_RUN} --height 768 --width 512 --batch-size 8 --cycles 1 --epochs-per-cycle 3 --no-average --time-budget-min {budget}

compare(f"{DRIVE_ROOT}/runs/{BIG_RUN}/checkpoints/best.pt", BIG_RUN)

[prepare] cache name: atr_768x512 | local: /content/data/atr_768x512 | Drive archive: /content/drive/MyDrive/clothseg/data/cache/atr_768x512.tar
[prepare] free local disk: 189.0 GB
[prepare] no cache found. Building it from the raw dataset (first run only).
[download] copying raw ATR from Drive: /content/drive/MyDrive/clothseg/data/raw/atr
[storage] copied train-00000-of-00002-f3a663f7140ee7fd.parquet (375.7 MB) -> /content/data/raw/atr in 6s
[storage] copied train-00001-of-00002-74610e243c32d5d6.parquet (384.0 MB) -> /content/data/raw/atr in 9s
[preprocess] 17706 samples in 2 parquet file(s); target fits inside 768x512 (HxW)
[preprocess] 4352/17706 (25%)  216 img/s  ETA 62s
[preprocess] 8981/17706 (51%)  223 img/s  ETA 39s
[preprocess] 13589/17706 (77%)  225 img/s  ETA 18s
[preprocess] processed 17706 samples in 78s; 0 failed
[preprocess] cache ready: /content/data/atr_768x512
[preprocess] {"height": 768, "width": 512, "seed": 42, "num_samples": 17706, "num_failed": 0, "num_exact_dupl

In [7]:
import os
import pandas as pd

# Learning-rate test: SegFormer-B3 for 12 epochs with half and with double the current learning rate,
# everything else as in the 12-epoch model r1_segformer_b3. Rule fixed in PLAN.md, section 28.1.
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
RUNS = {"lr_half": ("3e-5", "3e-4"), "lr_double": ("1.2e-4", "1.2e-3")}
for name, (enc, head) in RUNS.items():
    print("=" * 100)
    print(name, "| encoder lr", enc, "| decoder lr", head)
    budget = int(minutes_left())
    !python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name abl_segformer_b3_{name} --model segformer_b3 --height {HEIGHT} --width {WIDTH} --epochs 12 --encoder-lr {enc} --head-lr {head} --time-budget-min {budget}

COLS = ["garment_miou", "miou", "iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress", "iou_belt", "iou_scarf", "val_loss"]
CORE = ["iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress"]
base = pd.read_csv(f"{DRIVE_ROOT}/runs/r1_segformer_b3/val_log.csv").set_index("epoch").loc[12, COLS]
table, unfinished = {"lr 6e-5 (current)": base}, []
for name in RUNS:
    path = f"{DRIVE_ROOT}/runs/abl_segformer_b3_{name}/val_log.csv"
    done = len(pd.read_csv(path)) if os.path.exists(path) else 0
    if done < 12:
        unfinished.append(f"{name}: {done} of 12 epochs")
    else:
        table[name] = pd.read_csv(path).set_index("epoch").loc[12, COLS]

print()
print("validation at epoch 12")
print(pd.DataFrame(table).round(4).to_string())
print()
for name in RUNS:
    if name in table:
        gain = table[name]["garment_miou"] - base["garment_miou"]
        worst = min(table[name][c] - base[c] for c in CORE)
        ok = gain >= 0.005 and worst >= -0.01
        print(f"{name}: garment mIoU change {100 * gain:+.2f} points; largest drop on a core garment {100 * worst:+.2f} points -> "
              + ("MEETS the rule" if ok else "does NOT meet the rule") + " (at least +0.50, and no core garment below -1.00)")
if unfinished:
    print("NOT FINISHED:", "; ".join(unfinished), "-> run this cell again to continue those runs.")


lr_half | encoder lr 3e-5 | decoder lr 3e-4
[checkpoint] loaded step_00004008.pt
config.json: 100% 70.0k/70.0k [00:00<00:00, 82.7MB/s]
[model] segformer_b3: 47.2 M parameters (encoder 44.1 M, head 3.2 M) | 15 classes | weights not initialised from ImageNet
[train] RESUMED from epoch 5, batch 0/1002 (global step 4008/12024); best mIoU so far 0.7390; 46 min trained in earlier sessions; model weights restored from the checkpoint
[train] segformer_b3 | 15 classes | 576x384 | batch 16 | fp16 True | NVIDIA L4 | 8 workers
[train] data: train 16038, val 690 images (fingerprint 8b4d7a60ccb0) | 1002 steps/epoch
[train] schedule: 12 epochs = 12024 steps; this command runs until the end of epoch 12; time budget 598 min; checkpoint every 10 min
[train] epoch 5/12 step 42/1002 | loss 0.2137 (ce 0.1343, lovasz 0.2931) | lr 1.99e-05 | 13.5 img/s | about 158 min left for this command (plus validation)
[train] epoch 5/12 step 92/1002 | loss 0.2145 (ce 0.1403, lovasz 0.2887) | lr 1.98e-05 | 24.1 img/s | 